# 05 - Distillation math notes

This notebook keeps the main training idea visible. The student is not only copying final teacher text; it is learning from the teacher's softened token distribution at each response position.

For one token position, the project uses:

```text
L = α · T² · KL(p_teacher^T || p_student^T) + (1 - α) · CE(y, p_student)
```

`T` softens the distributions. Larger values reveal more of the teacher's ranking among plausible next tokens. `α` controls the balance between imitation and the original hard label.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

report_dir = Path("../reports")
report_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
def softmax(logits: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    scaled = logits / temperature
    scaled = scaled - scaled.max()
    exp = np.exp(scaled)
    return exp / exp.sum()


teacher_logits = np.array([7.0, 5.8, 4.2, 3.9, 2.0, 0.5])
tokens = ["yes", "likely", "maybe", "unclear", "no", "other"]

temperature_rows = []
for temperature in [1.0, 2.0, 4.0]:
    probabilities = softmax(teacher_logits, temperature)
    temperature_rows.extend(
        {"temperature": temperature, "token": token, "probability": probability}
        for token, probability in zip(tokens, probabilities)
    )

temperature_frame = pd.DataFrame(temperature_rows)
temperature_frame.pivot(index="token", columns="temperature", values="probability").round(3)


In [ ]:
ax = temperature_frame.pivot(index="token", columns="temperature", values="probability").plot(
    kind="bar",
    figsize=(8, 4),
)
ax.set_title("Teacher distribution becomes softer as temperature increases")
ax.set_ylabel("probability")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(report_dir / "temperature_softening.png", dpi=160)


In [ ]:
hard_loss = 2.4
soft_loss = 1.7
alpha_rows = []
for alpha in np.linspace(0.0, 1.0, 6):
    alpha_rows.append(
        {
            "alpha": round(float(alpha), 2),
            "combined_loss": alpha * soft_loss + (1 - alpha) * hard_loss,
            "hard_weight": 1 - alpha,
            "soft_weight": alpha,
        }
    )

pd.DataFrame(alpha_rows).round(3)


In [ ]:
def cache_size_estimate(rows: int, sequence_length: int, vocab_size: int, top_k: int) -> dict[str, float]:
    positions = rows * max(sequence_length - 1, 0)
    full_logits_mb = positions * vocab_size * 2 / 1_048_576
    topk_mb = positions * top_k * (2 + 4) / 1_048_576
    return {
        "full_fp16_logits_mb": full_logits_mb,
        "topk_logits_and_indices_mb": topk_mb,
        "compression_ratio": full_logits_mb / topk_mb,
    }


cache_size_estimate(rows=320, sequence_length=384, vocab_size=50_257, top_k=32)
